In [ ]:
# @title Mise en place de l'environnement
%set_env MLFLOW_TRACKING_URI https://dagshub.com/m09/landscape-classifier.mlflow
%set_env MPLBACKEND notebook
!rm -rf sample_data .config
!pip install -U uv
!git config --global user.email "jeanne@durant.fr"
!git config --global user.name "Jeanne Durant"
!git config --global init.defaultBranch main
!git clone https://github.com/shuuchuu/landscape-classifier.git
%cd /content/landscape-classifier
!uv pip install --system -e ".[fastapi]"

# MLflow, registre de modèles & déploiement

## Identifiants pour le serveur MLflow

Entrez vos identifiants DagsHub (le mot de passe est le jeton secret, pas votre mot de passe DagsHub)&nbsp;:

In [ ]:
username = ""
token = ""

In [ ]:
# @title À exécuter
%set_env MLFLOW_TRACKING_USERNAME {username}
%set_env MLFLOW_TRACKING_PASSWORD {token}

## Récupération des données

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!mv landscape/seg_train train-data

## Publication d'un modèle sur un registre de modèles

Adaptez le fichier `src/landscape_classifier/train.py` pour entraîner un modèle en enregistrant les métriques avec [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html). Les pages sur [Keras](https://mlflow.org/docs/latest/python_api/mlflow.keras.html) de l'API Python MLflow & de la [flavor Keras](https://mlflow.org/docs/latest/models.html#keras-keras) des modèles MLflow pourront être utiles.

Publiez ensuite le modèle appris dans le registre de modèles MLflow en fin d'exécution, sous le nom `dev.ml.landscape-classifier`. [Cette page](https://mlflow.org/docs/latest/models.html#keras-keras) en parle ainsi que [celle-ci](https://mlflow.org/docs/latest/model-registry.html#adding-an-mlflow-model-to-the-model-registry).

La cellule suivante lance l'entraînement&nbsp;:

In [ ]:
!landscape-classifier

## Utilisation du registre de modèles MLflow

En utilisant l'API MLflow, effectuez les tâches suivantes sur le registre de modèles.

### Création d'un client MLflow

Créez un client MLflow et affichez l'adresse de tracking pour vérifier qu'elle est bien récupérée depuis la variable d'environnement définie en haut du notebook.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
import mlflow

MODEL_NAME = "dev.ml.landscape-classifier"

client = mlflow.MlflowClient()
client.tracking_uri

### Recherche de modèles

Affichez tous les modèles enregistrés et pour chacun des modèles la version créée la plus récemment.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
def latest_version(name: str) -> str:
  versions = client.search_model_versions(f"name = '{name}'")
  return str(max(int(v.version) for v in versions))


for registered_model in client.search_registered_models():
  print(registered_model.name, latest_version(registered_model.name))

### Récupération d'un modèle depuis le registre de modèles

Maintenant que vous savez récupérer depuis Python le nom et la version des modèles présents sur le registre, récupérez-en un et vérifiez que c'est bien l'objet que vous anticipez.

Il y a plusieurs manières de charger un modèle si celui-ci dispose de plusieurs *flavors*. Qu'est-ce que cela change&nbsp;?

In [ ]:
# Votre code ici

#### Solution

In [ ]:
from pathlib import Path

version = latest_version(MODEL_NAME)
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{version}")
print(list(model.metadata.flavors))
model.predict([Path("forest1.jpg").read_bytes(), Path("forest2.jpg").read_bytes()])

Le modèle de la solution est publié comme un modèle `pyfunc` personnalisé&nbsp;: sa seule flavor est `python_function`, qui donne tout le pipeline (prétraitement des images, prédiction, mise en forme du résultat). Un modèle publié avec la flavor Keras se charge aussi comme un modèle Keras (`mlflow.keras.load_model`), qui ne donne que le réseau, sans le prétraitement.

### Ajout d'alias et d'étiquettes

Les alias et les étiquettes (tags) permettent d'organiser les modèles dans le registre de modèles MLflow. Les alias ciblent forcément une version en particulier alors que les étiquettes peuvent aussi cibler toutes les versions d'un modèle.

Ajoutez un alias `champion` à la dernière version de votre modèle, un tag `domain` à la valeur `cv` pour votre modèle et un tag `size` à la valeur `small` pour la dernière version de votre modèle.

In [ ]:
# Votre code ici

Récupérez maintenant votre modèle en utilisant l'alias plutôt que la version exacte.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
client.set_registered_model_alias(MODEL_NAME, "champion", version)
client.set_registered_model_tag(MODEL_NAME, "domain", "cv")
client.set_model_version_tag(MODEL_NAME, version, "size", "small")

In [ ]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}@champion")
model.predict([Path("forest1.jpg").read_bytes()])

### Changement de phase d'un modèle

Quand vous souhaitez promouvoir un modèle dans votre chaîne de mise en production, la méthode recommandée actuellement est de copier le modèle vers un nouveau nom de modèle.

Copiez la version `champion` de votre modèle vers un nouveau nom, où le préfixe `dev.` est remplacé par `staging.`.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
client.copy_model_version(
  src_model_uri=f"models:/{MODEL_NAME}@champion",
  dst_name="staging.ml.landscape-classifier",
)

## Création d'une API avec FastAPI

Toutes les questions de cette partie sont à coder dans le fichier `src/landscape_classifier/api.py`.

### Création d'une fonction de prétraitement adaptée à l'inférence

Créez une fonction pour charger une image dans le même format que celui utilisé pendant l'entraînement. Quel est le point auquel il faut faire extrêmement attention à ce stade&nbsp;?

### Création d'une fonction de chargement de modèle

Créez une fonction qui récupère le modèle entraîné au préalable depuis le registre de modèles MLflow. Cette fonction prendra en entrée l'URI du modèle à récupérer.

### Création d'une classe de retour FastAPI

FastAPI utilise l'excellente bibliothèque [Pydantic](https://docs.pydantic.dev/latest/) pour gérer les types d'entrée et de sortie des requêtes HTTP.

Créez une classe pour modéliser le type de retour avec la bibliothèque Pydantic. La classe de retour devra contenir a minima la classe prédite (celle avec la probabilité maximale), et un dictionnaire des différentes classes et leur probabilité selon le modèle.

### Création de l'API FastAPI

Implémentez l'API d'inférence à l'aide d'une méthode POST qui acceptera un fichier d'image. Vous pourrez vous aider de [cette page de documentation](https://fastapi.tiangolo.com/tutorial/request-files/).

## Création d'un Dockerfile

Écrivez un Dockerfile qui&nbsp;:

- utilise l'image de base Python 3.12&nbsp;;
- installe `uv` comme montré en cours&nbsp;;
- installe les dépendances du projet&nbsp;;
- copie les fichiers du projet&nbsp;;
- met en place un point d'entrée qui lance le serveur FastAPI.

Docker n'est pas disponible dans Colab&nbsp;: sur une machine avec Docker, on construit puis lance l'image en lui passant les identifiants MLflow en variables d'environnement&nbsp;:

    docker build -t shuuchuu/landscape-classifier .
    docker run --rm -it -e MLFLOW_TRACKING_URI -e MLFLOW_TRACKING_USERNAME \
      -e MLFLOW_TRACKING_PASSWORD -p 8000:80 shuuchuu/landscape-classifier

La documentation de l'API, qui sert aussi de client de test, est alors à l'adresse <http://localhost:8000/docs>.

## Amélioration de l'API

Comment modifier l'API pour qu'elle soit plus adaptée à la mise en production par image Docker&nbsp;?

*Votre réponse ici.*

## Solution

Pour toutes les questions qui n'ont pas de solution, consultez [la branche `solution` du dépôt GitHub](https://github.com/shuuchuu/landscape-classifier/tree/solution) de ces travaux pratiques.